# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access metadata fields
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets, their fields, and IDs.

Below, we enumerate all record sets with their `@id`s and their respective fields. Use these `@id`s for further data extraction.

In [ ]:
# List all record sets and their fields
recordsets = dataset.record_sets
print(f"Number of record sets: {len(recordsets)}\n")

recordsets_overview = []
for record_set in recordsets:
    print(f"Record set name: {record_set.name}")
    print(f"  @id: {record_set.id}")
    print(f"  Description: {record_set.description}")
    # Now list the fields (columns)
    print("  Fields:")
    for field in record_set.fields:
        print(f"    - {field.name} (id: {field.id}, type: {field.data_type})")
    print()
    recordsets_overview.append({'name': record_set.name, 'id': record_set.id})

if len(recordsets_overview) == 0:
    print('No record sets found in the metadata.')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. 
Below, we list out all available record set `@id`s for reference and load their data if present (using their `@id`).

In [ ]:
# Gather all record set @ids
record_set_ids = [rs['id'] for rs in recordsets_overview]
print(f"Available record set @id's: {record_set_ids}")

dataframes = {}

for record_set_id in record_set_ids:
    print(f"\nExtracting records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records and {len(df.columns)} columns.")
        print(f"Columns: {df.columns.tolist()}")
        display(df.head())
    else:
        print(f"No records found for record set: {record_set_id}")

# Select a particular record set for further example analysis:
if len(record_set_ids) > 0 and record_set_ids[0] in dataframes:
    main_record_set_id = record_set_ids[0]
    df = dataframes[main_record_set_id]
    print(f"\nMain DataFrame (first record set): {main_record_set_id}\nShape: {df.shape}")
    display(df.head())
else:
    print("No record set with tabular data found for analysis.")

## 4. Exploratory Data Analysis (EDA)
Let's select a numeric field from the main DataFrame for demonstrative filtering, normalization, and basic grouping. All fields and columns should always be referenced using their `@id` as previously listed.

In [ ]:
# Pick a numeric field, for example, 'coverage' if present (replace FIELD_ID below with the @id)

if 'main_record_set_id' in globals():
    df = dataframes[main_record_set_id]

    # Try to find a likely numeric column by inspecting columns for likely candidates
    numeric_candidates = [col for col in df.columns if df[col].dtype.kind in 'iufc' or 'coverage' in col or 'count' in col or 'abundance' in col or 'mw' in col.lower() or 'pi' in col.lower()]
    print(f"Numeric candidates: {numeric_candidates}")
    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]
        print(f"Using numeric field '@id': {numeric_field_id}\n")

        # Filter records
        # Set a dynamic threshold (e.g., mean for demonstration)
        try:
            threshold = df[numeric_field_id].mean()
        except Exception:
            threshold = 0

        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df[[numeric_field_id]].head())

        # Normalize the chosen field
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by another field if possible
        group_candidates = [col for col in df.columns if col != numeric_field_id and df[col].nunique() < min(10, len(df)//4)]
        if group_candidates:
            group_field_id = group_candidates[0]
            print(f"\nGrouping by field: {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by {group_field_id} (showing mean {numeric_field_id}):")
            display(grouped_df.head())
        else:
            print("No suitable categorical group field found.")
    else:
        print("No numeric field could be automatically selected for EDA.")
else:
    print("No suitable DataFrame found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Adjust the field names to match those present in your DataFrame.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'main_record_set_id' in globals() and numeric_candidates:
    plt.figure(figsize=(7, 4))
    field_to_plot = numeric_field_id
    df_to_plot = df[[field_to_plot]].dropna()
    sns.histplot(df_to_plot[field_to_plot], kde=True, bins=30)
    plt.title(f"Distribution of {field_to_plot}")
    plt.xlabel(field_to_plot)
    plt.ylabel("Frequency")
    plt.show()

    # If there's a grouping field
    if 'group_field_id' in locals():
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=group_field_id, y=field_to_plot, data=df)
        plt.title(f"{field_to_plot} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(field_to_plot)
        plt.show()
else:
    print("No suitable data found for visualization.")

## 6. Conclusion
In this notebook, we demonstrated how to explore and process the `Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells` dataset using the `mlcroissant` library. We loaded metadata, identified available record sets and their fields (by `@id`), extracted data, and completed basic exploratory data analysis, including normalization and visualization.

For advanced usage, you may chain more sophisticated transformations and analysis pipelines, referencing all entities by their `@id` as per FAIR and Croissant best practices.